# Feature Engineering — Dataset Final

**Objetivo:** Crear el dataset definitivo para modelado.

En este notebook:
1. Cargamos las 3 tablas raw (application, bureau, previous_application)
2. Creamos las features definitivas desde las tablas secundarias
3. Merge central
4. Features derivadas de negocio (ratios, interacciones)
5. Guardamos el dataset final para preprocessing

**Regla:** Cada feature debe tener una justificación de negocio clara.

---
## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Setup listo.')

---
## 2. Carga de datos raw

In [ ]:
app = pd.read_csv('../data/raw/application_train.csv')
bureau = pd.read_csv('../data/raw/bureau.csv')
prev = pd.read_csv('../data/raw/previous_application.csv')

print(f'application_train: {app.shape[0]:,} filas x {app.shape[1]} columnas')
print(f'bureau:            {bureau.shape[0]:,} filas x {bureau.shape[1]} columnas')
print(f'previous_application: {prev.shape[0]:,} filas x {prev.shape[1]} columnas')

---
## 3. Features desde bureau

Justificación de negocio: El historial crediticio externo indica si el cliente es propenso a defaultear.

In [ ]:
# === AGREGACIONES NUMÉRICAS ===
bureau_num = bureau.groupby('SK_ID_CURR').agg({
    'SK_ID_BUREAU': 'count',
    'AMT_CREDIT_SUM': ['sum', 'mean', 'max'],
    'AMT_CREDIT_SUM_DEBT': ['sum', 'mean'],
    'AMT_CREDIT_SUM_OVERDUE': 'sum',
    'AMT_CREDIT_MAX_OVERDUE': 'max',
    'AMT_ANNUITY': ['mean', 'max'],
    'DAYS_CREDIT': ['mean', 'min', 'max'],
    'CREDIT_DAY_OVERDUE': ['mean', 'max'],
    'DAYS_CREDIT_UPDATE': 'mean',
    'CNT_CREDIT_PROLONG': 'sum',
})

bureau_num.columns = ['_'.join(col).strip() for col in bureau_num.columns]
bureau_num = bureau_num.reset_index()

# Renombrar para claridad
bureau_num = bureau_num.rename(columns={
    'SK_ID_BUREAU_count': 'bureau_total_credits',
    'AMT_CREDIT_SUM_sum': 'bureau_total_credit_sum',
    'AMT_CREDIT_SUM_mean': 'bureau_avg_credit',
    'AMT_CREDIT_SUM_max': 'bureau_max_credit',
    'AMT_CREDIT_SUM_DEBT_sum': 'bureau_total_debt',
    'AMT_CREDIT_SUM_DEBT_mean': 'bureau_avg_debt',
    'AMT_CREDIT_SUM_OVERDUE_sum': 'bureau_total_overdue',
    'AMT_CREDIT_MAX_OVERDUE_max': 'bureau_max_overdue',
    'DAYS_CREDIT_mean': 'bureau_avg_days_credit',
    'DAYS_CREDIT_min': 'bureau_oldest_credit_days',
    'DAYS_CREDIT_max': 'bureau_newest_credit_days',
    'CREDIT_DAY_OVERDUE_mean': 'bureau_avg_overdue_days',
    'CREDIT_DAY_OVERDUE_max': 'bureau_max_overdue_days',
})

print(f'Features numéricas de bureau: {bureau_num.shape[1] - 1}')
bureau_num.head()

In [ ]:
# === PROPORCIONES DE ESTADO ===
bureau_status = pd.get_dummies(bureau[['SK_ID_CURR', 'CREDIT_ACTIVE']])
bureau_status = bureau_status.groupby('SK_ID_CURR').sum().reset_index()

# Calcular proporciones
status_cols = [c for c in bureau_status.columns if c.startswith('CREDIT_ACTIVE_')]
total = bureau_status[status_cols].sum(axis=1)
for col in status_cols:
    bureau_status[f'{col}_pct'] = bureau_status[col] / total

print(f'Proporciones de estado: {len(status_cols)} categorías')

In [ ]:
# === FEATURES DERIVADAS DE NEGOCIO ===
bureau_derived = pd.DataFrame()
bureau_derived['SK_ID_CURR'] = bureau['SK_ID_CURR'].unique()

# Conteo de estados por cliente
refused = bureau.groupby('SK_ID_CURR')['CREDIT_ACTIVE'].apply(lambda x: (x == 'Bad debt').sum()).reset_index(name='bureau_bad_debt_count')
active = bureau.groupby('SK_ID_CURR')['CREDIT_ACTIVE'].apply(lambda x: (x == 'Active').sum()).reset_index(name='bureau_active_count')

bureau_derived = bureau_derived.merge(refused, on='SK_ID_CURR', how='left')
bureau_derived = bureau_derived.merge(active, on='SK_ID_CURR', how='left')
bureau_derived = bureau_derived.merge(bureau_num[['SK_ID_CURR', 'bureau_total_credits']], on='SK_ID_CURR', how='left')

# Ratios
bureau_derived['bureau_bad_debt_rate'] = bureau_derived['bureau_bad_debt_count'] / bureau_derived['bureau_total_credits']
bureau_derived['bureau_active_rate'] = bureau_derived['bureau_active_count'] / bureau_derived['bureau_total_credits']

print('Features derivadas de bureau:')
bureau_derived.head()

In [ ]:
# === UNIR TODO BUREAU ===
bureau_features = bureau_num.merge(bureau_status[['SK_ID_CURR'] + [f'{c}_pct' for c in status_cols]], 
                                   on='SK_ID_CURR', how='left')
bureau_features = bureau_features.merge(bureau_derived[['SK_ID_CURR', 'bureau_bad_debt_rate', 'bureau_active_rate']], 
                                        on='SK_ID_CURR', how='left')

# Ratios de negocio
bureau_features['bureau_debt_ratio'] = bureau_features['bureau_total_debt'] / bureau_features['bureau_total_credit_sum']
bureau_features['bureau_overdue_ratio'] = bureau_features['bureau_total_overdue'] / bureau_features['bureau_total_credit_sum']

print(f'Features de bureau listas: {bureau_features.shape[1] - 1} columnas')

---
## 4. Features desde previous_application

Justificación de negocio: El comportamiento previo del cliente dentro de Home Credit predice su comportamiento futuro.

In [ ]:
# === AGREGACIONES NUMÉRICAS ===
prev_num = prev.groupby('SK_ID_CURR').agg({
    'SK_ID_PREV': 'count',
    'AMT_APPLICATION': ['sum', 'mean', 'max'],
    'AMT_CREDIT': ['sum', 'mean', 'max'],
    'AMT_ANNUITY': ['mean', 'max'],
    'AMT_DOWN_PAYMENT': ['sum', 'mean'],
    'DAYS_DECISION': ['mean', 'min', 'max'],
    'CNT_PAYMENT': ['mean', 'max'],
    'RATE_DOWN_PAYMENT': 'mean',
})

prev_num.columns = ['_'.join(col).strip() for col in prev_num.columns]
prev_num = prev_num.reset_index()

prev_num = prev_num.rename(columns={
    'SK_ID_PREV_count': 'prev_total_applications',
    'AMT_APPLICATION_sum': 'prev_total_requested',
    'AMT_APPLICATION_mean': 'prev_avg_requested',
    'AMT_APPLICATION_max': 'prev_max_requested',
    'AMT_CREDIT_sum': 'prev_total_approved',
    'AMT_CREDIT_mean': 'prev_avg_approved',
    'DAYS_DECISION_mean': 'prev_avg_decision_days',
    'DAYS_DECISION_min': 'prev_oldest_decision_days',
    'DAYS_DECISION_max': 'prev_newest_decision_days',
})

print(f'Features numéricas de previous: {prev_num.shape[1] - 1}')
prev_num.head()

In [ ]:
# === PROPORCIONES DE ESTADO ===
prev_status = pd.get_dummies(prev[['SK_ID_CURR', 'NAME_CONTRACT_STATUS']])
prev_status = prev_status.groupby('SK_ID_CURR').sum().reset_index()

status_cols_prev = [c for c in prev_status.columns if c.startswith('NAME_CONTRACT_STATUS_')]
total_prev = prev_status[status_cols_prev].sum(axis=1)
for col in status_cols_prev:
    prev_status[f'{col}_pct'] = prev_status[col] / total_prev

print(f'Proporciones de estado: {len(status_cols_prev)} categorías')

In [ ]:
# === FEATURES DERIVADAS DE NEGOCIO ===
prev_derived = pd.DataFrame()
prev_derived['SK_ID_CURR'] = prev['SK_ID_CURR'].unique()

refused_prev = prev.groupby('SK_ID_CURR')['NAME_CONTRACT_STATUS'].apply(lambda x: (x == 'Refused').sum()).reset_index(name='prev_refused_count')
approved_prev = prev.groupby('SK_ID_CURR')['NAME_CONTRACT_STATUS'].apply(lambda x: (x == 'Approved').sum()).reset_index(name='prev_approved_count')

prev_derived = prev_derived.merge(refused_prev, on='SK_ID_CURR', how='left')
prev_derived = prev_derived.merge(approved_prev, on='SK_ID_CURR', how='left')
prev_derived = prev_derived.merge(prev_num[['SK_ID_CURR', 'prev_total_applications']], on='SK_ID_CURR', how='left')

prev_derived['prev_reject_rate'] = prev_derived['prev_refused_count'] / prev_derived['prev_total_applications']
prev_derived['prev_approve_rate'] = prev_derived['prev_approved_count'] / prev_derived['prev_total_applications']

print('Features derivadas de previous_application:')
prev_derived.head()

In [ ]:
# === UNIR TODO PREVIOUS ===
prev_features = prev_num.merge(prev_status[['SK_ID_CURR'] + [f'{c}_pct' for c in status_cols_prev]], 
                               on='SK_ID_CURR', how='left')
prev_features = prev_features.merge(prev_derived[['SK_ID_CURR', 'prev_reject_rate', 'prev_approve_rate']], 
                                    on='SK_ID_CURR', how='left')

# Ratio de negocio: cuánto le dieron vs cuánto pidió
prev_features['prev_credit_ratio'] = prev_features['prev_total_approved'] / prev_features['prev_total_requested']

print(f'Features de previous listas: {prev_features.shape[1] - 1} columnas')

---
## 5. Merge central

Unimos todo a `application_train`.

In [ ]:
TARGET_COL = 'TARGET'
df = app.copy()

print(f'Start: {df.shape[1]} columnas')

# Merge bureau
df = df.merge(bureau_features, on='SK_ID_CURR', how='left')
print(f'Después de bureau: {df.shape[1]} columnas (+{df.shape[1] - app.shape[1]})')

# Merge previous_application
df = df.merge(prev_features, on='SK_ID_CURR', how='left')
print(f'Después de previous: {df.shape[1]} columnas (+{df.shape[1] - app.shape[1]})')

print(f'\nDataset final: {df.shape[0]:,} filas x {df.shape[1]} columnas')

---
## 6. Features derivadas de negocio

Creaciones basadas en lógica de negocio que combinan information de múltiples tablas.

In [ ]:
# === RATIOS DE NEGOCIO ===

# Cuánto del crédito solicitado le aprobaron
df['credit_approval_ratio'] = df['AMT_CREDIT'] / df['AMT_GOODS_PRICE'].replace(0, np.nan)

# Cuánto paga mensualmente vs su ingreso
df['annuity_to_income'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL'].replace(0, np.nan)

# Cuánto debe vs cuánto gana
df['credit_to_income'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL'].replace(0, np.nan)

# Ratio de bienes al crédito
df['goods_to_credit'] = df['AMT_GOODS_PRICE'] / df['AMT_CREDIT'].replace(0, np.nan)

print('Ratios de negocio creados.')

In [ ]:
# === FEATURES DE TIEMPO ===

# Edad en años (DAYS_BIRTH es negativo)
df['age_years'] = (-df['DAYS_BIRTH'] / 365.25).astype(float)

# Antigüedad laboral en años
df['employment_years'] = (-df['DAYS_EMPLOYED'] / 365.25).astype(float)

# Ratio antigüedad laboral / edad
df['employment_ratio'] = df['employment_years'] / df['age_years'].replace(0, np.nan)

# Días desde última modificación de registro
df['registration_years'] = (-df['DAYS_REGISTRATION'] / 365.25).astype(float)

print('Features de tiempo creadas.')

In [ ]:
# === FEATURES DE EXTERNAL SOURCE ===
# Promedio de las 3 fuentes externas (si existen)
ext_cols = [c for c in df.columns if c.startswith('EXT_SOURCE_')]
if ext_cols:
    df['ext_source_mean'] = df[ext_cols].mean(axis=1)
    df['ext_source_std'] = df[ext_cols].std(axis=1)
    df['ext_source_min'] = df[ext_cols].min(axis=1)
    df['ext_source_max'] = df[ext_cols].max(axis=1)
    print(f'Features de external source creadas: 4 (usando {ext_cols})')
else:
    print('No se encontraron columnas EXT_SOURCE_')

In [ ]:
# === FEATURES DE DOCUMENTOS ===
doc_cols = [c for c in df.columns if c.startswith('FLAG_DOCUMENT_')]
df['total_documents'] = df[doc_cols].sum(axis=1)
print(f'Features de documentos creadas: 1 (total de {len(doc_cols)} documentos)')

In [ ]:
# === FEATURES DE CONTACTO ===
contact_cols = ['FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL']
contact_cols = [c for c in contact_cols if c in df.columns]
df['total_contact_methods'] = df[contact_cols].sum(axis=1)
print(f'Features de contacto creadas: 1 (total de {len(contact_cols)} métodos)')

---
## 7. Resumen del dataset final

In [ ]:
# Separar features por origen
app_cols = [c for c in app.columns]
bureau_only = [c for c in df.columns if c.startswith('bureau_')]
prev_only = [c for c in df.columns if c.startswith('prev_')]
engineered = [c for c in df.columns if c not in app_cols and c not in bureau_only and c not in prev_only]

print('RESUMEN DEL DATASET FINAL')
print('=' * 50)
print(f'Total columnas:   {df.shape[1]}')
print(f'  Originales:     {len(app_cols)}')
print(f'  De bureau:       {len(bureau_only)}')
print(f'  De previous:     {len(prev_only)}')
print(f'  Ingeniería:      {len(engineered)}')
print()
print(f'Total filas:      {df.shape[0]:,}')
print(f'Target:           {TARGET_COL}')
print(f'Ratio target:     {df[TARGET_COL].value_counts().to_dict()}')

In [ ]:
# Features de ingeniería creadas
print('FEATURES DE INGENIERÍA:')
for f in sorted(engineered):
    print(f'  - {f}')

In [ ]:
# Nulos en el dataset final
nulls = df.isnull().sum()
nulls_pct = (nulls / len(df) * 100).round(2)

null_report = pd.DataFrame({'nulos': nulls, 'pct': nulls_pct}).query('nulos > 0').sort_values('pct', ascending=False)
print(f'Columnas con nulos: {len(null_report)} de {df.shape[1]}')
print(f'\nTop 15:')
null_report.head(15)

---
## 8. Guardar dataset final

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

df.to_csv('../data/processed/application_train_features.csv', index=False)
print(f'Guardado: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(f'Ruta: data/processed/application_train_features.csv')